# Pathway 3: FOM Concatenation

This tutorial demonstrates the **per-domain Full Order Model concatenation** workflow. Each solid is solved independently, and the FOMs are cascaded via Kirchhoff coupling.

```
Multi Solid Model  →  FDS per Domain  →  Concat. of FOMs  →  Reduced Order Model
```

## 1. Create Multi-Domain Geometry

We create a rectangular waveguide and split it at the midpoint into two separate domains.

In [ ]:
import os
from geometry.importers import OCCImporter
from geometry.primitives import RectangularWaveguide

a = 100e-3  # Width: 100 mm
L = 200e-3  # Length: 200 mm
maxh = 0.04

# Option A: Use a STEP file with splitting plane
step_path = os.path.join('..', '..', 'examples', 'rwg_step_split', 'rectangular_waveguide.step')
if os.path.exists(step_path):
    geom = OCCImporter(step_path, unit='mm', auto_build=False, maxh=maxh)
    geom.add_splitting_plane_at_z(L / 2)
    geom.split()
    geom.finalize(maxh=maxh)
    print("Loaded and split STEP file")
else:
    # Option B: Use primitive (single domain)
    geom = RectangularWaveguide(a=a, L=L, maxh=maxh)
    print("Using single-domain primitive (split STEP not found)")
    print(f"  To use multi-domain, place STEP file at: {step_path}")

## 2. Set Up the Solver

In [ ]:
from solvers.frequency_domain import FrequencyDomainSolver

fds = FrequencyDomainSolver(geom, order=3)
fds.assemble_matrices(nmodes=1)
fds.print_info()

print(f"\nIs compound (multi-domain)? {fds.is_compound}")
print(f"Number of domains: {fds.n_domains}")
if fds.is_compound:
    print(f"External ports: {fds.ports}")
    print(f"All ports (incl. internal): {fds.all_ports}")

## 3. Solve Per-Domain FOMs

Each domain is solved independently. The solver automatically detects the compound structure.

In [ ]:
results = fds.solve(fmin=1.5, fmax=3.0, nsamples=30, store_snapshots=True)
print(f"Solved at {len(fds.frequencies)} frequency points")

## 4. Concatenate FOMs

The concatenation step enforces **Kirchhoff coupling** at internal ports:

$$
\mathbf{a}_{\text{int}}^{(d)} = \mathbf{b}_{\text{int}}^{(d+1)}, \quad
\mathbf{b}_{\text{int}}^{(d)} = \mathbf{a}_{\text{int}}^{(d+1)}
$$

This cascades the per-domain S-matrices into a global 2-port response.

In [ ]:
if fds.is_compound:
    cs = fds.foms.concatenate()
    cs.couple()
    cs_results = cs.solve(fmin=1.5, fmax=3.0, nsamples=30)
    print(f"Concatenated system ports: {cs.ports}")
else:
    print("Single domain — concatenation not applicable.")

## 5. Reduce the Concatenated System

In [ ]:
from rom.reduction import ModelOrderReduction

if fds.is_compound:
    rom = cs.reduce(tol=1e-6)
    rom.solve(fmin=1.5, fmax=3.0, nsamples=500)
    rom.print_info()
else:
    rom = ModelOrderReduction(fds)
    rom.reduce(tol=1e-6)
    rom.solve(fmin=1.5, fmax=3.0, nsamples=500)

## 6. Plot Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

from analytical.rectangular_waveguide import RWGAnalytical

analytical = RWGAnalytical(a=a, L=L)
frequencies = np.linspace(1.5, 3.0, 200) * 1e9
S_ana = analytical.s_parameters(frequencies)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
freq_ghz_ana = frequencies / 1e9
freq_ghz_rom = rom.frequencies / 1e9

ax = axes[0]
ax.plot(freq_ghz_ana, 20*np.log10(np.abs(S_ana['S11'])+1e-15), '-', label='Analytical', lw=2)
ax.plot(freq_ghz_rom, rom.get_s_db(1, 1), '--', label='ROM', lw=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S11| (dB)')
ax.set_title('S11'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(freq_ghz_ana, 20*np.log10(np.abs(S_ana['S21'])+1e-15), '-', label='Analytical', lw=2)
ax.plot(freq_ghz_rom, rom.get_s_db(1, 2), '--', label='ROM', lw=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S21| (dB)')
ax.set_title('S21'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('FOM Concatenation vs Analytical', fontsize=14)
plt.tight_layout(); plt.show()

## 7. Access Per-Domain Results

In [ ]:
if fds.is_compound:
    for domain in fds.domains:
        domain_results = fds.get_domain_results(domain)
        print(f"Domain '{domain}':")
        print(f"  Ports: {domain_results['ports']}")
        print(f"  Z shape: {domain_results['Z'].shape}")

## Summary

In this tutorial we:
1. Created a multi-domain geometry by splitting at the midpoint
2. Solved each domain independently
3. Concatenated the FOMs via Kirchhoff coupling
4. Reduced the concatenated system for wideband analysis

**Next:** [Pathway 4: ROM Concatenation](pathway4_rom_concatenation.ipynb) — the most efficient workflow.